
# Phase 5 — Pretrained Model Integration

## Goal

Integrate Transkun, a pretrained piano audio-to-MIDI transcription model,
into Piano2MIDI through a Python wrapper.

We are performing inference, not training a neural network.

- **Training** adjusts model parameters using examples.
- **Inference** uses learned parameters to predict notes from new audio.
- **Evaluation** compares predictions with a reference transcription.

### Architecture

Audio file → Piano2MIDI wrapper → Transkun CLI → MIDI output

The wrapper validates inputs, invokes Transkun, handles execution failures,
and checks that an output file was produced. Transkun itself performs
the neural-network inference.

We keep the model behind a wrapper so that Piano2MIDI can eventually
replace the transcription backend without rewriting the whole application.


In [18]:
from pathlib import Path
from piano2midi.transcription.transkun_backend import transcribe

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

audio_path = project_root/'data'/'processed'/'piano_sample_20s.wav'
output_path = project_root/'data'/'processed'/'phase5_transcription.mid'

assert audio_path.is_file(), f'Audio sample not found: {audio_path}'

print('Input: ',audio_path)
print('Output: ', output_path)

Input:  c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\piano_sample_20s.wav
Output:  c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\phase5_transcription.mid



## Run inference

The following cell invokes the pretrained model through our wrapper.
It does not train the model or modify its weights.

The device is set to CUDA to use the NVIDIA GPU when available.
The wrapper raises an exception if Transkun fails or doesn't produce
a non-empty output file.


In [19]:
transkun_executable = (
    project_root
    / ".venv-transkun311"
    / "Scripts"
    / "transkun.exe"
)

assert transkun_executable.is_file(), (
    f"Transkun executable not found: {transkun_executable}"
)

midi_path = transcribe(audio_path,output_path,device='cuda',executable=str(transkun_executable))
print('Transcription complete: ', midi_path)
print('Output size: ', midi_path.stat().st_size, 'bytes')

Transcription complete:  c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\phase5_transcription.mid
Output size:  835 bytes


In [20]:
import mido

midi = mido.MidiFile(midi_path)

note_on_events = [
    message
    for track in midi.tracks
    for message in track
    if message.type == 'note_on' and message.velocity>0
]

print('Tracks: ',len(midi.tracks))
print('Ticks per beat: ',midi.ticks_per_beat)
print('Note-on events:',len(note_on_events))

assert midi_path.is_file() and midi_path.stat().st_size>0
assert len(note_on_events)>0, 'The generated MIDI contrains no note on events'

Tracks:  2
Ticks per beat:  960
Note-on events: 95


In [21]:
midi_files = [
    path for path in (project_root/'data').rglob('*')
    if path.is_file() and path.suffix.lower() in {'.mid','.midi'}
]

for path in midi_files:
    print(path)

c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\mini_piano_performance.mid
c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\phase5_transcription.mid
c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\piano_sample_20s.mid
c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\piano_sample_20s_reference.mid
c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\piano_sample_wrapper.mid
c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\sustain_test.mid
c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\test_c4.mid
c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\test_phrase.mid
c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\velocity_test.mid
c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\raw\maestro\piano_sample.midi


In [22]:
reference_midi_path = project_root / "data" / "raw" / "maestro" / "piano_sample.midi"

reference_midi = mido.MidiFile(reference_midi_path)

print("Reference tracks:", len(reference_midi.tracks))
print("Ticks per beat:", reference_midi.ticks_per_beat)
print("Reference duration (seconds):", reference_midi.length)

Reference tracks: 2
Ticks per beat: 384
Reference duration (seconds): 1797.4205729166665


In [23]:
from pathlib import Path
import mido

reference_source = project_root / "data" / "raw" / "maestro" / "piano_sample.midi"
reference_excerpt_path = project_root / "data" / "processed" / "piano_sample_20s_reference.mid"

source_midi = mido.MidiFile(reference_source)

# Keep the first 20 seconds of the original MIDI.
excerpt_seconds = 20.0
tempo = 500_000  # MIDI default tempo: 120 BPM
absolute_time = 0.0

excerpt = mido.MidiFile(ticks_per_beat=source_midi.ticks_per_beat)

for source_track in source_midi.tracks:
    new_track = mido.MidiTrack()
    excerpt.tracks.append(new_track)

    absolute_ticks = 0
    tempo = 500_000
    events = []

    for message in source_track:
        absolute_ticks += message.time

        if message.type == "set_tempo":
            tempo = message.tempo

        time_seconds = mido.tick2second(
            absolute_ticks, source_midi.ticks_per_beat, tempo
        )

        if time_seconds > excerpt_seconds:
            break

        events.append((absolute_ticks, message))

    previous_ticks = 0
    for ticks, message in events:
        new_track.append(message.copy(time=ticks - previous_ticks))
        previous_ticks = ticks

excerpt.save(reference_excerpt_path)

print("Saved reference excerpt:", reference_excerpt_path)
print("Duration:", excerpt.length, "seconds")

Saved reference excerpt: c:\Users\Deep\Desktop\Deep\Piano2MIDI\piano2midi\data\processed\piano_sample_20s_reference.mid
Duration: 19.986979166666664 seconds


In [24]:
reference_excerpt = mido.MidiFile(reference_excerpt_path)

print("Reference duration:", reference_excerpt.length, "seconds")
print("Reference tracks:", len(reference_excerpt.tracks))
print("Ticks per beat:", reference_excerpt.ticks_per_beat)
print(
    "Reference note-on events:",
    sum(
        1
        for track in reference_excerpt.tracks
        for msg in track
        if msg.type == "note_on" and msg.velocity > 0
    ),
)

Reference duration: 19.986979166666664 seconds
Reference tracks: 2
Ticks per beat: 384
Reference note-on events: 97


In [25]:
from piano2midi.evaluation.metrics import evaluate_notes

def midi_to_notes(path):
    midi = mido.MidiFile(path)
    notes = []
    tempo = 500_000

    for track in midi.tracks:
        absolute_ticks = 0
        active = {}

        for msg in track:
            absolute_ticks += msg.time

            if msg.type == "set_tempo":
                tempo = msg.tempo

            elif msg.type == "note_on" and msg.velocity > 0:
                active.setdefault(msg.note, []).append(absolute_ticks)

            elif msg.type == "note_off" or (
                msg.type == "note_on" and msg.velocity == 0
            ):
                if active.get(msg.note):
                    start = active[msg.note].pop(0)
                    notes.append({
                        "pitch": msg.note,
                        "onset": mido.tick2second(
                            start, midi.ticks_per_beat, tempo
                        ),
                        "offset": mido.tick2second(
                            absolute_ticks, midi.ticks_per_beat, tempo
                        ),
                    })

    return notes

ground_truth = midi_to_notes(reference_excerpt_path)
predictions = midi_to_notes(midi_path)

results = evaluate_notes(ground_truth, predictions)

print(f"Reference notes: {len(ground_truth)}")
print(f"Predicted notes: {len(predictions)}")
for key, value in results.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

Reference notes: 92
Predicted notes: 95
TP: 76
FP: 19
FN: 16
precision: 0.8000
recall: 0.8261
F1: 0.8128



## Results and conclusions

Transkun successfully generated a MIDI from the 20-second piano audio sample
through the Piano2MIDI Python wrapper.

The output contained 95 note-on events. Evaluation against the matching
reference excerpt produced:

- Precision: 0.8000
- Recall: 0.8261
- F1: 0.8128

The Windows result matches the earlier Linux experiment, providing evidence
that the inference pipeline behaves consistently across both environments.

### Limitations

- This is one short excerpt, not a comprehensive quality benchmark.
- The evaluation depends on correct MIDI parsing and note alignment.
- The pretrained model is used as-is; improving transcription accuracy is
  deferred to Phase 6.

### Phase 5 conclusion

The pretrained model is installed and can be invoked through a reusable
Python wrapper. Input validation, execution failure handling, output checks,
and focused automated tests are in place.
